# BioJEPA v0.6 Data Prep - Notebook 2: Perturbation Embeddings

This notebook handles:
1. Extracting unique perturbations per dataset
2. Linking perturbations to sequences (sgRNA protospacer, SMILES)
3. Mapping perturbations to target proteins (ENSG -> UniProt -> sequence)
4. Running embedding models (NucleotideTransformer, ESM-2, ChemMRL)
5. Saving feature banks with index mappings

**Inputs (from Notebook 1):**
- `gene_to_idx.json` - ENSG -> index for gene universe
- `dataset_splits.json` - train/val/test perturbation sets per dataset

**Outputs:**
- `pert_embd/seq_banks/dna_embeddings.npy` - [N_dna, 1536]
- `pert_embd/seq_banks/dna_to_idx.json` - sgID_AB -> idx
- `pert_embd/seq_banks/chemical_embeddings.npy` - [N_chem, 1024]
- `pert_embd/seq_banks/chemical_to_idx.json` - SMILES -> idx
- `pert_embd/target_banks/protein_targets.npy` - [N_genes, 320]
- `pert_embd/target_banks/gene_to_idx.json` - ENSG -> idx
- `pert_embd/input_to_id.json` - "GENE_mode_dataset" -> seq_idx (for pathway evals)

In [1]:
from pathlib import Path
from collections import defaultdict
from Bio import SeqIO, Entrez
from tqdm import tqdm
import pandas as pd
import numpy as np
import scanpy as sc
import mygene
import torch
import json
import gzip
import gc
import re
import pubchempy as pcp

Entrez.email = 'gptomics@gmail.com'

In [2]:
ref_dir = Path('/Users/djemec/data/jepa/reference_data')
data_dir = Path('/Users/djemec/data/jepa/v0_6')

seq_banks_dir = data_dir / 'pert_embd' / 'seq_banks'
target_banks_dir = data_dir / 'pert_embd' / 'target_banks'
seq_banks_dir.mkdir(parents=True, exist_ok=True)
target_banks_dir.mkdir(parents=True, exist_ok=True)

In [3]:
def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(1337)
        device = 'cuda'
    print(f'using {device}')
    return device
device = get_device()

using cpu


In [4]:
with open(data_dir / 'gene_to_idx.json') as f:
    gene_to_idx = json.load(f)

with open(data_dir / 'dataset_splits.json') as f:
    dataset_splits = json.load(f)

print(f'Loaded gene universe with {len(gene_to_idx)} genes')

Loaded gene universe with 12288 genes


In [5]:
def is_valid(val):
    if pd.isna(val):
        return False
    if isinstance(val, str) and (val.strip() == '' or val.strip().lower() == 'nan'):
        return False
    return True
def print_list_head(print_list, n=10):
    print({j:print_list[j] for j in list(print_list.keys())[:n]})

## Step 1: Load CRISPRi Reference Library

Maps sgID -> protospacer sequence for k562e_raw, rep1e, k562gw

In [6]:
crispr_df = pd.read_csv(ref_dir / 'crispr' / 'hcrispri_all.csv')
crispr_df['sgID_clean'] = crispr_df['sgID'].astype(str).str.replace(',', '-').str.strip()

crispr_df.head()

,sgID,gene,transcript,protospacer sequence,selection rank,predicted score,empirical score,off-target stringency,Sublibrary half,sgID_clean
0,A1BG_-_58858617.23-P1,A1BG,P1,GGAGACCCAGCGCTAACCAG,1.0,1.008816,NaN,0,Top5,A1BG_-_58858617.23-P1
1,A1BG_-_58858788.23-P1,A1BG,P1,GGGGCACCCAGGAGCGGTAG,2.0,0.901176,NaN,0,Top5,A1BG_-_58858788.23-P1
2,A1BG_+_58858964.23-P1,A1BG,P1,GCTCCGGGCGACGTGGAGTG,3.0,0.836188,NaN,0,Top5,A1BG_+_58858964.23-P1
3,A1BG_-_58858630.23-P1,A1BG,P1,GAACCAGGGGTGCCCAAGGG,4.0,0.827551,NaN,0,Top5,A1BG_-_58858630.23-P1
4,A1BG_+_58858549.23-P1,A1BG,P1,GGCGAGGAACCGCCCAGCAA,5.0,0.775395,NaN,0,Top5,A1BG_+_58858549.23-P1


In [7]:
sgid_to_seq = dict(zip(crispr_df['sgID_clean'], crispr_df['protospacer sequence'].str.strip()))
sgid_to_gene = dict(zip(crispr_df['sgID_clean'], crispr_df['gene'].str.strip()))

print(f'Loaded {len(sgid_to_seq)} sgID -> protospacer mappings')

Loaded 267846 sgID -> protospacer mappings


## Step 2: Extract Unique sgID_AB from CRISPRi Datasets

k562e_raw, rep1e, k562gw all use dual-guide CRISPRi

In [8]:
datasets = {
    'k562e_raw': ref_dir / 'raw_k562e' / 'K562_essential_raw_singlecell_01.h5ad',
    'rep1e': ref_dir / 'rep1e' / 'rpe1_raw_singlecell_01.h5ad',
    'k562gw': ref_dir / 'k562gw' / 'K562_gwps_raw_singlecell_01.h5ad',
}

In [9]:
def get_sequences_from_sgid_ab(sgid_ab):
    if not is_valid(sgid_ab):
        return None, None
    sgid_ab = str(sgid_ab).replace(',', '-').strip()
    parts = sgid_ab.split('|')
    seq_a = sgid_to_seq.get(parts[0].strip())
    seq_b = sgid_to_seq.get(parts[1].strip()) if len(parts) > 1 else None
    return seq_a, seq_b

def get_gene_from_sgid_ab(sgid_ab):
    if not is_valid(sgid_ab):
        return None
    sgid_ab = str(sgid_ab).replace(',', '-').strip()
    parts = sgid_ab.split('|')
    return sgid_to_gene.get(parts[0].strip())

In [10]:
crispri_sgids = {}  # dataset -> set of unique sgID_AB
sgid_ab_to_gene = {}  # sgID_AB -> target gene name

for ds_name, ds_path in datasets.items():
    print(f'Loading {ds_name}...')
    adata = sc.read_h5ad(ds_path, backed='r')
    
    if 'sgID_AB' in adata.obs.columns:
        sgids = set(adata.obs['sgID_AB'].dropna().unique())
    else:
        print(f'No sgID_AB column, {ds_name}')
        sgids = set()
    
    crispri_sgids[ds_name] = sgids
    print(f'Found {len(sgids)} unique sgID_AB entries')
    
    for sgid in sgids:
        if sgid not in sgid_ab_to_gene:
            gene = get_gene_from_sgid_ab(sgid)
            if gene:
                sgid_ab_to_gene[sgid] = gene

Loading k562e_raw...
Found 2273 unique sgID_AB entries
Loading rep1e...
Found 2662 unique sgID_AB entries
Loading k562gw...
Found 11187 unique sgID_AB entries


In [11]:
all_crispri_sgids = set()
for ds_sgids in crispri_sgids.values():
    all_crispri_sgids.update(ds_sgids)

print(f'Total unique sgID_AB across CRISPRi datasets: {len(all_crispri_sgids)}')

Total unique sgID_AB across CRISPRi datasets: 11256


## Step 3: Load Adamson Protospacer Mapping

In [12]:
adamson_df = pd.read_csv(ref_dir / 'adamson' / 'adamson_protospacer_sgrna.csv')
adamson_gene_to_protospacer = dict(zip(adamson_df['Gene'].str.strip(), adamson_df['Protospacer'].str.strip()))

print_list_head(adamson_gene_to_protospacer)
print(f'Loaded {len(adamson_gene_to_protospacer)} Adamson gene -> protospacer mappings')

{'AARS': 'GAGGGCGGCCTACCTCTCCT', 'AMIGO3/GMPPB': 'GGGGCCAGCAGCCGTCTACC', 'ARHGAP22': 'GGTCCGTCCGGAGCCAGGAG', 'ASCC3': 'GCGCACAGACCCGGCGAGGA', 'ATF6': 'GGGGATCTGAGAATGTACCA', 'ATP5B': 'GAGTCTCCGCAAGGCCCCGG', 'CAD': 'GTAGGAGCCTCGGGCGCGCT', 'CARS': 'GAGCCATGGCAGATTCCTCC', 'CCND3': 'GCGACGTCCGAGCATTCCA', 'CHERP': 'GCGCTGGTGGTCGATCGTG'}
Loaded 85 Adamson gene -> protospacer mappings


In [13]:
adamson_path = ref_dir / 'adamson' / 'AdamsonWeissman2016_GSM2406681_10X010.h5ad'
adata = sc.read_h5ad(adamson_path, backed='r')

In [14]:
if 'perturbation' in adata.obs.columns:
    adamson_perts = set(adata.obs['perturbation'].dropna().unique())
else:
    adamson_perts = set()

adamson_perts = {p for p in adamson_perts if is_valid(p) and str(p).lower() not in ['control', 'ctrl', 'nan']}
print(f'Adamson unique perturbations: {len(adamson_perts)}')

Adamson unique perturbations: 114


## Step 4: Load Norman sgRNA and Target Mappings

In [15]:
norman_sgrna_df = pd.read_csv(ref_dir / 'norman' / 'norman_sgrna.csv')
norman_target_df = pd.read_csv(ref_dir / 'norman' / 'norman_guideid_ensemble_id_map.csv')

norman_sgrna_df.head()

,number,gene_A,gene_B,protospacer_sequence_A,protospacer_sequence_B,GBC,Notes
0,1,AHR,NegCtrl0,GAGACGGAATGGAATCCAGA,GTCGCGCCCGCTCCAGGGAC,GAGTCGGACTCCGCCATG,NaN
1,2,ARID1A,NegCtrl0,GCCGCCTGGCAAACCCGGAG,GTCGCGCCCGCTCCAGGGAC,CACAGCATACTAGCGACC,NaN
2,3,ARRDC3,NegCtrl0,GGTACAGTAGGTGTAGAGCT,GTCGCGCCCGCTCCAGGGAC,AAGTTTGAGCGATGCCGT,NaN
3,4,ATL1,NegCtrl0,GAGTGCTCGGGCGGGCCGCT,GTCGCGCCCGCTCCAGGGAC,GCTAAGGGTTTGATGAGG,NaN
4,5,BAK1,NegCtrl0,GCAGGCAGGGCGGCTGTCAG,GTCGCGCCCGCTCCAGGGAC,TCAGACGTGGTGAGATCG,NaN


In [16]:
norman_target_df.head()

,guide_id,UMI_count,num_cells,first_target,first_id,control_first_expr,first_expr,fold_first_expr,second_target,second_id,control_second_expr,second_expr,fold_second_expr,ks_de,fitness,guide_UMI_count,guide_read_count,guide_coverage
0,AHR_FEV,8636.844697,264,AHR,ENSG00000106546,0.003264,0.613636,188.002841,FEV,ENSG00000163497,0.375493,5.174242,13.779864,3261,-0.577088,29.715909,644.553030,21.459278
1,AHR_KLF1,15603.354370,412,AHR,ENSG00000106546,0.003264,0.332524,101.877124,KLF1,ENSG00000105610,0.962464,4.614078,4.794025,379,-0.037193,45.427184,866.148058,19.279608
2,AHR_NegCtrl0,11797.929020,479,AHR,ENSG00000106546,0.003264,0.471816,144.552714,NegCtrl0,NaN,NaN,NaN,NaN,1345,-0.333858,46.966597,1025.187891,21.693670
3,ARID1A_NegCtrl0,10933.016480,182,ARID1A,ENSG00000117713,0.926697,0.961538,1.037598,NegCtrl0,NaN,NaN,NaN,NaN,1548,-0.319112,74.593407,1577.384615,22.406428
4,ARRDC3_NegCtrl0,13836.148150,405,ARRDC3,ENSG00000113369,0.052360,0.214815,4.102684,NegCtrl0,NaN,NaN,NaN,NaN,99,-0.126237,63.824691,1255.207407,19.590217


In [17]:
norman_gene_to_protospacer_a = dict(zip(norman_sgrna_df['gene_A'].str.strip(), norman_sgrna_df['protospacer_sequence_A'].str.strip()))
print_list_head(norman_gene_to_protospacer_a)

{'AHR': 'GAGACGGAATGGAATCCAGA', 'ARID1A': 'GCCGCCTGGCAAACCCGGAG', 'ARRDC3': 'GGTACAGTAGGTGTAGAGCT', 'ATL1': 'GAGTGCTCGGGCGGGCCGCT', 'BAK1': 'GCAGGCAGGGCGGCTGTCAG', 'BCL2L11': 'GAGGCTCGGACAGGTAAAGG', 'BCORL1': 'GGATCGCTGAGAGGACCGAG', 'BPGM': 'GCCTACTCCCGGAACAGGAG', 'C19orf26': 'GGTCCCTGGGACCTCAGGAC', 'C3orf72': 'GCTCTGGCGGAGCTGCCTCC'}


In [18]:
norman_gene_to_protospacer_b = {}
for _, row in norman_sgrna_df.iterrows():
    if is_valid(row['gene_B']) and str(row['gene_B']).lower() != 'negctrl0':
        norman_gene_to_protospacer_b[row['gene_B'].strip()] = row['protospacer_sequence_B'].strip()

print_list_head(norman_gene_to_protospacer_b)

{'BAK1': 'GCAGGCAGGGCGGCTGTCAG', 'C19orf26': 'GGTCCCTGGGACCTCAGGAC', 'CBFA2T3': 'GACGCTGTAGGGCTCAGGGT', 'CDKN1A': 'GAGCCTGGCCGAGTTCCAGC', 'CDKN1B': 'GAGCTCGCTAGGAGCCGGGG', 'CEBPA': 'GGCAGCCTCGGGATACTCCT', 'CEBPB': 'GCGGCGGCAGGGCGCAGCGG', 'CEBPE': 'GCCCCTCAAAAAACAAACCC', 'CLDN6': 'GACCCTGTCCCTGACGGCAG', 'CNN1': 'GAGGCCCAATGGACAGTGGG'}


In [19]:
norman_gene_to_ensg = {}
for _, row in norman_target_df.iterrows():
    if is_valid(row.get('first_target')) and is_valid(row.get('first_id')):
        norman_gene_to_ensg[row['first_target'].strip()] = row['first_id'].strip()
    if is_valid(row.get('second_target')) and is_valid(row.get('second_id')):
        norman_gene_to_ensg[row['second_target'].strip()] = row['second_id'].strip()

print(f'Norman gene -> protospacer_A: {len(norman_gene_to_protospacer_a)}')
print(f'Norman gene -> protospacer_B: {len(norman_gene_to_protospacer_b)}')
print(f'Norman gene -> ENSG: {len(norman_gene_to_ensg)}')

Norman gene -> protospacer_A: 108
Norman gene -> protospacer_B: 49
Norman gene -> ENSG: 105


In [20]:
norman_path = ref_dir / 'norman' / 'NormanWeissman2019_filtered.h5ad'
adata = sc.read_h5ad(norman_path, backed='r')

In [21]:
if 'guide_id' in adata.obs.columns:
    norman_guide_ids = set(adata.obs['guide_id'].dropna().unique())
else:
    norman_guide_ids = set()

norman_guide_ids = {g for g in norman_guide_ids if is_valid(g) and str(g).lower() not in ['control', 'ctrl', 'nan']}
print(f'Norman unique guide_ids: {len(norman_guide_ids)}')

Norman unique guide_ids: 290


## Step 5: Load Sciplex Chemical Perturbations

In [24]:
sciplex_path = ref_dir / 'sciplex' / 'SrivatsanTrapnell2020_sciplex3.h5ad'
adata = sc.read_h5ad(sciplex_path, backed='r')

In [26]:
if 'perturbation' in adata.obs.columns:
    sciplex_drugs = set(adata.obs['perturbation'].dropna().unique())
else:
    sciplex_drugs = set()

sciplex_drugs = {d for d in sciplex_drugs if is_valid(d) and str(d).lower() not in ['control', 'vehicle', 'dmso', 'nan']}
print(f'Sciplex unique drugs: {len(sciplex_drugs)}')

Sciplex unique drugs: 188


In [27]:
list(sciplex_drugs)[:10]

['ABT-737',
 'Fulvestrant',
 'Tubastatin A HCl',
 'NVP-BSK805 2HCl',
 'Lomustine ',
 'Busulfan ',
 'Trametinib (GSK1120212)',
 'Alendronate sodium trihydrate',
 'Streptozotocin (STZ)',
 'Altretamine']

## Step 6: Map Drugs to SMILES via ChEMBL

We use the chembl_36_chemreps.txt file which contains SMILES for ChEMBL compounds. We'll also use PubChemPy as a fallback.

In [30]:
def get_smiles_robust(raw_name):
    clean_parts = re.split(r'[()\[\]?]', raw_name)
    candidates = [raw_name] + [p.strip() for p in clean_parts if p.strip()]
    candidates = list(dict.fromkeys(candidates))

    for candidate in candidates:
        try:
            compounds = pcp.get_compounds(candidate, 'name')
            if compounds:
                return compounds[0].smiles
        except Exception:
            continue
    return None

In [31]:
drug_to_smiles = {}
missing_drugs = []
for drug in tqdm(sciplex_drugs, desc='Fetching SMILES from PubChem'):
    smiles = get_smiles_robust(drug)
    if smiles:
        drug_to_smiles[drug] = smiles
    else:
        missing_drugs.append(drug)

print_list_head(drug_to_smiles)
print(f'Found SMILES for {len(drug_to_smiles)} / {len(sciplex_drugs)} drugs')
if missing_drugs:
    print(f'Missing SMILES for: {missing_drugs}')

Fetching SMILES from PubChem: 100%|█████████████████████████████████████████████| 188/188 [01:09<00:00,  2.72it/s]

{'ABT-737': 'CN(C)CC[C@H](CSC1=CC=CC=C1)NC2=C(C=C(C=C2)S(=O)(=O)NC(=O)C3=CC=C(C=C3)N4CCN(CC4)CC5=CC=CC=C5C6=CC=C(C=C6)Cl)[N+](=O)[O-]', 'Fulvestrant': 'C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2O)[C@@H](CC4=C3C=CC(=C4)O)CCCCCCCCCS(=O)CCCC(C(F)(F)F)(F)F', 'Tubastatin A HCl': 'CN1CCC2=C(C1)C3=CC=CC=C3N2CC4=CC=C(C=C4)C(=O)NO.Cl', 'NVP-BSK805 2HCl': 'C1CNCCC1N2C=C(C=N2)C3=NC4=C(C=CC=C4N=C3)C5=CC(=C(C(=C5)F)CN6CCOCC6)F.Cl.Cl', 'Lomustine ': 'C1CCC(CC1)NC(=O)N(CCCl)N=O', 'Busulfan ': 'CS(=O)(=O)OCCCCOS(=O)(=O)C', 'Trametinib (GSK1120212)': 'CC1=C2C(=C(N(C1=O)C)NC3=C(C=C(C=C3)I)F)C(=O)N(C(=O)N2C4=CC=CC(=C4)NC(=O)C)C5CC5', 'Alendronate sodium trihydrate': 'C(CC(O)(P(=O)(O)O)P(=O)(O)[O-])CN.O.O.O.[Na+]', 'Streptozotocin (STZ)': 'CN(C(=O)N[C@@H]1[C@H]([C@@H]([C@H](O[C@@H]1O)CO)O)O)N=O', 'Altretamine': 'CN(C)C1=NC(=NC(=N1)N(C)C)N(C)C'}
Found SMILES for 188 / 188 drugs


In [32]:
with open(data_dir / 'pert_embd' / 'drug_to_smiles.json', 'w') as f:
    json.dump(drug_to_smiles, f, indent=2)

## Step 7: Load UniProt Protein Sequences

Map ENSG -> UniProt accession -> protein sequence

In [33]:
uniprot_fasta = ref_dir / 'uniprot' / 'UP000005640_9606.fasta.gz'

acc_to_seq = {}
with gzip.open(uniprot_fasta, 'rt') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        parts = record.id.split('|')
        if len(parts) >= 2:
            acc_to_seq[parts[1]] = str(record.seq)

print(f'Loaded {len(acc_to_seq)} UniProt sequences')

Loaded 20659 UniProt sequences


In [34]:
all_target_genes = set()

for sgid, gene in sgid_ab_to_gene.items():
    all_target_genes.add(gene)

for gene in adamson_perts:
    gene_clean = str(gene).split('/')[0].strip() if '/' in str(gene) else str(gene).strip()
    all_target_genes.add(gene_clean)

for gene in norman_gene_to_ensg.keys():
    all_target_genes.add(gene)

print(f'Total unique target genes: {len(all_target_genes)}')
list(all_target_genes)[:10]

Total unique target genes: 10008


['ORC2',
 'PRKAR2A',
 'NCOR2',
 'SAP18',
 'TAC3',
 'UFM1',
 'KANSL3',
 'ZNF526',
 'MADD',
 'MAPK12']

In [35]:
gene_to_ensg = {}
ensg_to_gene = {}

with open(data_dir / 'gene_names.json') as f:
    gene_names = json.load(f)

for i, (ensg, idx) in enumerate(gene_to_idx.items()):
    if ensg.startswith('ENSG'):
        name = gene_names[idx] if idx < len(gene_names) else ensg
        gene_to_ensg[name] = ensg
        ensg_to_gene[ensg] = name

print(f'Gene -> ENSG mappings: {len(gene_to_ensg)}')

Gene -> ENSG mappings: 12248


In [36]:
target_ensg_ids = set()
genes_without_ensg = []

for gene in all_target_genes:
    if gene in gene_to_ensg:
        target_ensg_ids.add(gene_to_ensg[gene])
    elif gene.startswith('ENSG'):
        target_ensg_ids.add(gene)
    else:
        genes_without_ensg.append(gene)

for ensg in norman_gene_to_ensg.values():
    if is_valid(ensg):
        target_ensg_ids.add(ensg)

print(f'Initial genes missing ENSG: {len(genes_without_ensg)}')
print(f'Target ENSG IDs from direct mapping: {len(target_ensg_ids)}')

Initial genes missing ENSG: 1410
Target ENSG IDs from direct mapping: 8620


In [ ]:
mg = mygene.MyGeneInfo()
if genes_without_ensg:
    mg_results = mg.querymany(genes_without_ensg, s
                              copes='ensembl.gene,symbol,alias,entrezgene,reporter,uniprot', 
                              fields='uniprot,symbol,ensembl.gene', species='human')
    found_by_symbol = 0
    still_missing = []
    
    for res in mg_results:
        query_ensg = res['query']
        if 'uniprot' in res:
            # Prefer Swiss-Prot (Reviewed) -> TrEMBL (Unreviewed)
            if 'Swiss-Prot' in res['uniprot']:
                val = res['uniprot']['Swiss-Prot']
                # Handle list vs string
                ensg_to_acc[query_ensg] = val[0] if isinstance(val, list) else val
            elif 'TrEMBL' in res['uniprot']:
                val = res['uniprot']['TrEMBL']
                ensg_to_acc[query_ensg] = val[0] if isinstance(val, list) else val

In [37]:
mg = mygene.MyGeneInfo()
if genes_without_ensg:
    print(f'Searching mygene for {len(genes_without_ensg)} missing genes by symbol/alias...')
    symbol_results = mg.querymany(genes_without_ensg, scopes='symbol,alias', fields='ensembl.gene', species='human', verbose=False)
    
    found_by_symbol = 0
    still_missing = []
    for res in symbol_results:
        if 'ensembl' in res:
            ensg_val = res['ensembl']
            if isinstance(ensg_val, list):
                ensg = ensg_val[0].get('gene') if isinstance(ensg_val[0], dict) else ensg_val[0]
            elif isinstance(ensg_val, dict):
                ensg = ensg_val.get('gene')
            else:
                ensg = ensg_val
            
            if ensg and isinstance(ensg, str) and ensg.startswith('ENSG'):
                target_ensg_ids.add(ensg)
                gene_to_ensg[res['query']] = ensg
                found_by_symbol += 1
            else:
                still_missing.append(res['query'])
        else:
            still_missing.append(res['query'])
    
    print(f'Found {found_by_symbol} additional ENSG IDs via symbol/alias search')
    print(f'Still missing: {len(still_missing)} genes')

print(f'Final target ENSG IDs to embed: {len(target_ensg_ids)}')

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


Searching mygene for 1410 missing genes by symbol/alias...
Found 1368 additional ENSG IDs via symbol/alias search
Still missing: 122 genes
Final target ENSG IDs to embed: 9928


In [40]:
still_missing

['PSMD12_pDS008',
 'SOCS1_pDS479',
 'IARS2_pDS091',
 'IDH3A_pDS393',
 'MRGBP_pDS124',
 'HSPA5_pDS371',
 'PDIA6_pDS029',
 'FARSB_pDS390',
 'negative_control',
 'SLMO2_pDS433',
 'CHERP_pDS024',
 'DHDDS_pDS383',
 'XRN1_pDS411',
 'ASCC3_pDS051',
 'GBF1_pDS044',
 'TTI1_pDS407',
 'SYVN1_pDS442',
 'ATF6_pBA586',
 'EIF2AK3_pBA573',
 'IER3IP1_pDS003',
 'DNAJC19_pDS026',
 'XBP1_pBA578',
 'IARS2_pDS090',
 'MRPL39_pDS498',
 'PSMD4_pDS488',
 'IER3IP1_pDS002',
 'COPB1_pDS065',
 'FCGR2C',
 'ATF4_pBA608',
 'SRP72_pDS505',
 'DERL2_pDS042',
 'AARS_pDS381',
 'TIMM23_pDS284',
 'COPZ1_pDS462',
 'YIPF5_pDS186',
 'SAMM50_pDS156',
 'ATF4_pBA576',
 'C7orf26_pDS004',
 'MARS_pDS394',
 'SEC61A1_pDS032',
 'RP5-862P8.2',
 'SEC61B_pDS162',
 'SARS_pDS467',
 'XBP1_pBA579',
 'TMEM167A_pDS038',
 'SCYL1_pDS160',
 '63(mod)_pBA580',
 'PSMA1_pDS007',
 'SLC35B1_pDS046',
 'SPCS3_pDS402',
 'TMED10_pDS036',
 'DDOST_pDS382',
 'P4HB_pDS397',
 'MTHFD1_pDS395',
 'DNAJC19_pDS074',
 'TELO2_pDS496',
 'SEC61G_pDS440',
 'CCND3_pDS006',


In [ ]:
mg = mygene.MyGeneInfo()

results = mg.querymany(list(target_ensg_ids), scopes='ensembl.gene', fields='uniprot,symbol', species='human', verbose=False)

ensg_to_uniprot = {}
no_uniprot = []
for res in results:
    if 'uniprot' in res:
        if 'Swiss-Prot' in res['uniprot']:
            val = res['uniprot']['Swiss-Prot']
            ensg_to_uniprot[res['query']] = val[0] if isinstance(val, list) else val
        elif 'TrEMBL' in res['uniprot']:
            val = res['uniprot']['TrEMBL']
            ensg_to_uniprot[res['query']] = val[0] if isinstance(val, list) else val
    else:
        no_uniprot.append(res['query'])

print(f'ENSG -> UniProt mappings: {len(ensg_to_uniprot)}')
print(f'No UniProt found for: {len(no_uniprot)} ENSG IDs')

In [ ]:
ensg_to_protein_seq = {}
missing_proteins = []

for ensg in target_ensg_ids:
    acc = ensg_to_uniprot.get(ensg)
    if acc and acc in acc_to_seq:
        ensg_to_protein_seq[ensg] = acc_to_seq[acc]
    else:
        missing_proteins.append(ensg)

print(f'Found protein sequences for {len(ensg_to_protein_seq)} / {len(target_ensg_ids)} targets')
print(f'Missing: {len(missing_proteins)}')

In [ ]:
if missing_proteins:
    print(f'Fetching {len(missing_proteins)} protein sequences from Entrez...')
    entrez_found = 0
    entrez_failed = []
    
    for ensg_id in tqdm(missing_proteins, desc='Entrez protein lookup'):
        try:
            search_handle = Entrez.esearch(db='gene', term=ensg_id, retmax=1)
            search_record = Entrez.read(search_handle)
            
            if search_record['IdList']:
                ncbi_gene_id = search_record['IdList'][0]
                link_handle = Entrez.elink(dbfrom='gene', db='protein', id=ncbi_gene_id, linkname='gene_protein_refseq')
                link_record = Entrez.read(link_handle)
                
                if link_record and link_record[0]['LinkSetDb']:
                    protein_id = link_record[0]['LinkSetDb'][0]['Link'][0]['Id']
                    fetch_handle = Entrez.efetch(db='protein', id=protein_id, rettype='fasta', retmode='text')
                    seq_record = SeqIO.read(fetch_handle, 'fasta')
                    ensg_to_protein_seq[ensg_id] = str(seq_record.seq)
                    entrez_found += 1
                else:
                    entrez_failed.append(ensg_id)
            else:
                entrez_failed.append(ensg_id)
        except Exception as e:
            print(f'Failed Entrez lookup for {ensg_id}: {e}')
            entrez_failed.append(ensg_id)
    
    print(f'Found {entrez_found} protein sequences via Entrez')
    print(f'Still missing: {len(entrez_failed)} proteins')
else:
    print('No missing proteins - skipping Entrez lookup')

## Step 8: Generate DNA Embeddings (NucleotideTransformer)

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [ ]:
nt_model_name = 'InstaDeepAI/NTv3_650M_pre'
nt_tokenizer = AutoTokenizer.from_pretrained(nt_model_name, trust_remote_code=True)
nt_model = AutoModelForMaskedLM.from_pretrained(nt_model_name, trust_remote_code=True)
nt_model.to(device).eval()

print(f'Loaded NucleotideTransformer: {nt_model_name}')

In [ ]:
def get_dna_embedding(sequence):
    inputs = nt_tokenizer([sequence], return_tensors='pt', padding='max_length', max_length=128, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    attention_mask = (inputs['input_ids'] != nt_tokenizer.pad_token_id).long()
    
    with torch.no_grad():
        outputs = nt_model(**inputs, output_hidden_states=True)
    
    embeddings = outputs.hidden_states[-1]
    mask = attention_mask.unsqueeze(-1).expand(embeddings.size()).float()
    pooled = (embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return pooled.squeeze(0).cpu().numpy()

In [ ]:
dna_vectors = []
dna_to_idx = {}
sgid_ab_to_dna_idx = {}
crispri_skipped = []

for sgid_ab in tqdm(all_crispri_sgids, desc='Embedding CRISPRi sgRNAs'):
    seq_a, seq_b = get_sequences_from_sgid_ab(sgid_ab)
    
    if seq_a is None:
        crispri_skipped.append(sgid_ab)
        continue
    
    emb_a = get_dna_embedding(seq_a)
    
    if seq_b is not None:
        emb_b = get_dna_embedding(seq_b)
        combined = (emb_a + emb_b) / 2
    else:
        combined = emb_a
    
    idx = len(dna_vectors)
    dna_vectors.append(combined)
    dna_to_idx[sgid_ab] = idx
    sgid_ab_to_dna_idx[sgid_ab] = idx

print(f'Generated {len(dna_vectors)} CRISPRi DNA embeddings')
print(f'Skipped {len(crispri_skipped)} / {len(all_crispri_sgids)} (no protospacer in reference)')

In [ ]:
adamson_gene_to_dna_idx = {}
adamson_skipped = []

for gene in tqdm(adamson_perts, desc='Embedding Adamson sgRNAs'):
    gene_clean = str(gene).split('/')[0].strip() if '/' in str(gene) else str(gene).strip()
    protospacer = adamson_gene_to_protospacer.get(gene_clean) or adamson_gene_to_protospacer.get(gene)
    
    if protospacer is None or not is_valid(protospacer):
        adamson_skipped.append(gene)
        continue
    
    emb = get_dna_embedding(protospacer)
    idx = len(dna_vectors)
    dna_vectors.append(emb)
    adamson_gene_to_dna_idx[gene] = idx

print(f'Generated {len(adamson_gene_to_dna_idx)} / {len(adamson_perts)} Adamson DNA embeddings')
print(f'Skipped {len(adamson_skipped)} (no protospacer mapping): {adamson_skipped[:10]}{"..." if len(adamson_skipped) > 10 else ""}')
print(f'Total DNA embeddings: {len(dna_vectors)}')

In [ ]:
norman_guide_to_dna_idx = {}

for _, row in tqdm(norman_sgrna_df.iterrows(), total=len(norman_sgrna_df), desc='Embedding Norman sgRNAs'):
    gene_a = row['gene_A']
    gene_b = row['gene_B']
    
    if not is_valid(gene_a):
        continue
    
    guide_id = f"{gene_a}_{gene_b}" if is_valid(gene_b) and str(gene_b).lower() != 'negctrl0' else gene_a
    
    seq_a = row['protospacer_sequence_A']
    seq_b = row['protospacer_sequence_B'] if is_valid(row.get('protospacer_sequence_B')) else None
    
    if not is_valid(seq_a):
        continue
    
    emb_a = get_dna_embedding(seq_a)
    
    if seq_b is not None and is_valid(seq_b) and str(gene_b).lower() != 'negctrl0':
        emb_b = get_dna_embedding(seq_b)
        combined = (emb_a + emb_b) / 2
    else:
        combined = emb_a
    
    idx = len(dna_vectors)
    dna_vectors.append(combined)
    norman_guide_to_dna_idx[guide_id] = idx

print(f'Generated {len(norman_guide_to_dna_idx)} Norman DNA embeddings')
print(f'Total DNA embeddings: {len(dna_vectors)}')

In [ ]:
del nt_model, nt_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
dna_embeddings = np.stack(dna_vectors, axis=0).astype(np.float32)
print(f'DNA embeddings shape: {dna_embeddings.shape}')

np.save(seq_banks_dir / 'dna_embeddings.npy', dna_embeddings)

with open(seq_banks_dir / 'dna_to_idx.json', 'w') as f:
    json.dump(dna_to_idx, f)

with open(seq_banks_dir / 'adamson_gene_to_dna_idx.json', 'w') as f:
    json.dump(adamson_gene_to_dna_idx, f)

with open(seq_banks_dir / 'norman_guide_to_dna_idx.json', 'w') as f:
    json.dump(norman_guide_to_dna_idx, f)

## Step 9: Generate Protein Target Embeddings (ESM-2)

In [ ]:
from transformers import AutoTokenizer, AutoModel

In [ ]:
esm_model_name = 'facebook/esm2_t6_8M_UR50D'
esm_tokenizer = AutoTokenizer.from_pretrained(esm_model_name)
esm_model = AutoModel.from_pretrained(esm_model_name)
esm_model.to(device).eval()

print(f'Loaded ESM-2: {esm_model_name}')

In [ ]:
def get_protein_embedding(protein_seq):
    inputs = esm_tokenizer(protein_seq, return_tensors='pt', truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = esm_model(**inputs)
    
    emb = outputs.last_hidden_state[0].mean(dim=0)
    return emb.cpu().numpy()

In [ ]:
protein_vectors = []
ensg_to_target_idx = {}

for ensg, protein_seq in tqdm(ensg_to_protein_seq.items(), desc='Embedding target proteins'):
    emb = get_protein_embedding(protein_seq)
    idx = len(protein_vectors)
    protein_vectors.append(emb)
    ensg_to_target_idx[ensg] = idx

print(f'Generated {len(protein_vectors)} protein target embeddings')

In [ ]:
del esm_model, esm_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
protein_embeddings = np.stack(protein_vectors, axis=0).astype(np.float32)
print(f'Protein embeddings shape: {protein_embeddings.shape}')

np.save(target_banks_dir / 'protein_targets.npy', protein_embeddings)

with open(target_banks_dir / 'gene_to_idx.json', 'w') as f:
    json.dump(ensg_to_target_idx, f)

## Step 10: Generate Chemical Embeddings (ChemMRL)

In [ ]:
chem_model_name = 'Derify/ChemMRL'
chem_tokenizer = AutoTokenizer.from_pretrained(chem_model_name)
chem_model = AutoModel.from_pretrained(chem_model_name)
chem_model.to(device).eval()

print(f'Loaded ChemMRL: {chem_model_name}')

In [ ]:
def get_chemical_embedding(smiles):
    inputs = chem_tokenizer(smiles, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = chem_model(**inputs)
    
    emb = outputs.last_hidden_state[0].mean(dim=0)
    return emb.cpu().numpy()

In [ ]:
chemical_vectors = []
smiles_to_idx = {}
drug_to_chem_idx = {}

for drug, smiles in tqdm(drug_to_smiles.items(), desc='Embedding chemicals'):
    if smiles in smiles_to_idx:
        drug_to_chem_idx[drug] = smiles_to_idx[smiles]
        continue
    
    emb = get_chemical_embedding(smiles)
    idx = len(chemical_vectors)
    chemical_vectors.append(emb)
    smiles_to_idx[smiles] = idx
    drug_to_chem_idx[drug] = idx

print(f'Generated {len(chemical_vectors)} chemical embeddings')

In [ ]:
del chem_model, chem_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
if chemical_vectors:
    chemical_embeddings = np.stack(chemical_vectors, axis=0).astype(np.float32)
    print(f'Chemical embeddings shape: {chemical_embeddings.shape}')
    
    np.save(seq_banks_dir / 'chemical_embeddings.npy', chemical_embeddings)
    
    with open(seq_banks_dir / 'chemical_to_idx.json', 'w') as f:
        json.dump(smiles_to_idx, f)
    
    with open(seq_banks_dir / 'drug_to_chem_idx.json', 'w') as f:
        json.dump(drug_to_chem_idx, f)
else:
    print('No chemical embeddings generated')

## Step 11: Build input_to_id.json for Pathway Evals

Maps "GENE_mode_dataset" -> seq_idx for pathway analysis

In [ ]:
input_to_id = {}

for sgid_ab, idx in sgid_ab_to_dna_idx.items():
    gene = sgid_ab_to_gene.get(sgid_ab)
    if gene:
        for ds in ['k562e', 'rep1e', 'k562gw']:
            key = f'{gene}_crispri_{ds}'
            if key not in input_to_id:
                input_to_id[key] = idx

for gene, idx in adamson_gene_to_dna_idx.items():
    gene_clean = str(gene).split('/')[0].strip() if '/' in str(gene) else str(gene).strip()
    key = f'{gene_clean}_crispri_adamson'
    input_to_id[key] = idx

for guide_id, idx in norman_guide_to_dna_idx.items():
    key = f'{guide_id}_crispra_norman'
    input_to_id[key] = idx

for drug, idx in drug_to_chem_idx.items():
    key = f'{drug}_inhibitor_sciplex'
    input_to_id[key] = idx

print(f'Built input_to_id with {len(input_to_id)} entries')

In [ ]:
with open(data_dir / 'pert_embd' / 'input_to_id.json', 'w') as f:
    json.dump(input_to_id, f)

## Step 12: Save Gene-to-Target Mappings for Training

In [ ]:
gene_to_target_idx = {}

for gene, ensg in gene_to_ensg.items():
    if ensg in ensg_to_target_idx:
        gene_to_target_idx[gene] = ensg_to_target_idx[ensg]

for ensg, idx in ensg_to_target_idx.items():
    gene_to_target_idx[ensg] = idx

for gene, ensg in norman_gene_to_ensg.items():
    if ensg in ensg_to_target_idx:
        gene_to_target_idx[gene] = ensg_to_target_idx[ensg]

print(f'Gene to target idx: {len(gene_to_target_idx)} mappings')

In [ ]:
with open(target_banks_dir / 'gene_to_target_idx.json', 'w') as f:
    json.dump(gene_to_target_idx, f)

## Summary

In [ ]:
print('=== Data Prep Notebook 2 Complete ===')
print(f'\nDNA Embeddings: {dna_embeddings.shape}')
print(f'  - CRISPRi sgID_AB: {len(sgid_ab_to_dna_idx)}')
print(f'  - Adamson genes: {len(adamson_gene_to_dna_idx)}')
print(f'  - Norman guides: {len(norman_guide_to_dna_idx)}')
print(f'\nProtein Embeddings: {protein_embeddings.shape}')
print(f'  - Target genes: {len(ensg_to_target_idx)}')
if chemical_vectors:
    print(f'\nChemical Embeddings: {chemical_embeddings.shape}')
    print(f'  - Unique SMILES: {len(smiles_to_idx)}')
    print(f'  - Drugs: {len(drug_to_chem_idx)}')
print(f'\ninput_to_id entries: {len(input_to_id)}')

In [ ]:
print('\nOutput files:')
for f in seq_banks_dir.glob('*'):
    print(f'  {f.name}')
for f in target_banks_dir.glob('*'):
    print(f'  {f.name}')